[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/6_optimizacion/_assets/02_simplex.ipynb)

# Notebook 2 · El sello

Acompaña a la **clase 2** de la unidad de modelado y optimización. Da por leídas
las cinco páginas.

Las páginas tabulan; esto calcula. Aquí la computadora rehace desde cero lo que
la clase dejó escrito a mano: **los ocho vértices del poliedro con lo que se
acaba en cada uno**, **quién es vecino de quién**, y **las trazas de simplex**,
para que ninguna de esas tablas te llegue como un acto de fe.

Después viene lo que ninguna página dibuja: **el valor del mejor plan contra las
horas de impresora, de 8 a 14**, que tiene dos codos.

Lo que aquí no está, a propósito: los precios sombra a diferencias finitas y el
codo de la energía. Eso ya corrió en el notebook 1, y repetirlo sería gastar una
clase en algo publicado.

Cierra con una bitácora nueva —el taller— que **no está resuelta en ninguna
página**. Ése es el ejercicio, y consiste en escribir tres datos. No hay nada que
entregar.

Corre las celdas en orden con `Shift + Enter`.

In [ ]:
# === Celda 1 · Preparación =====================================
# La convención de signos otra vez, porque es la fuente número uno de errores al
# pasar del papel al código. Los dos modelos de esta clase son de MÁXIMO:
#
#   - En las páginas la constante va a la derecha y todo está con <=.
#   - Al solver se le entrega TODO con <=:  a·x >= b  se escribe  -a·x <= -b.
#   - linprog MINIMIZA. Un máximo se resuelve con -c, y al valor que devuelve
#     hay que cambiarle el signo OTRA VEZ.
#   - Las cotas sobre una variable van en `bounds`; aquí basta x >= 0, que es lo
#     que linprog toma por omisión.

import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.optimize import linprog

TOL = 1e-9        # nada de ==: el solver devuelve 40.99999999999999 donde va 41

# Los planes se imprimen así en todo el notebook. El 0.0 forzado quita los -0.
fmt = lambda v: '(' + ', '.join(
    f'{0.0 if abs(t) < TOL else t:g}' for t in np.ravel(v)) + ')'

print('listo:', fmt([8, 2]), 'se lee así, y', fmt([-0.0, 4.5]), 'también')

## 1 · Los tres datos, en dos y en tres variables

*El modelo como matriz* dice que un problema lineal es exactamente una terna
$(c, A, b)$. Escribirla es todo el trabajo: de ahí en adelante el código no sabe
de impresoras.

$$
c=(4,3),\quad A=\begin{pmatrix}1&1\\2&1\\1&2\end{pmatrix},\quad b=\begin{pmatrix}10\\18\\18\end{pmatrix}
\qquad\qquad
c=(4,3,5),\quad A=\begin{pmatrix}1&1&1\\2&1&2\\1&2&3\end{pmatrix},\quad b=\begin{pmatrix}10\\18\\18\end{pmatrix}
$$

La matriz no se teclea de corrido: se arma **por columnas**, porque una columna
es la receta de una pieza y así viene la tabla de la bitácora. El modelo con
sello se escribe entonces con una línea más que el de la clase 1, y eso es todo
lo que costó la tercera pieza.

In [ ]:
# === Celda 2 · La terna, armada por columnas ===================
# Un renglón por recurso y una columna por pieza: es la tabla de *Cuando se acaba
# el dibujo*, sin la columna de disponible ni el renglón de créditos.

recursos = ['horas', 'polímero', 'energía']
receta   = {'filtro': [1, 2, 1],       # una hora, dos kilos, un kWh
            'celda':  [1, 1, 2],
            'sello':  [1, 2, 3]}
credito  = {'filtro': 4, 'celda': 3, 'sello': 5}

def terna(piezas, disponible):
    """Los tres datos, con las piezas en el orden en que se nombran."""
    c = np.array([credito[p] for p in piezas], float)
    A = np.column_stack([receta[p] for p in piezas]).astype(float)
    b = np.array(disponible, float)
    # La prueba de tamaños de *El modelo como matriz*: b tiene una entrada por
    # renglón y c una por columna.
    assert A.shape == (len(b), len(c)), A.shape
    return c, A, b

c2, A2, b2 = terna(['filtro', 'celda'],          [10, 18, 18])
c3, A3, b3 = terna(['filtro', 'celda', 'sello'], [10, 18, 18])

print('dos piezas   c =', c2, '  b =', b2, '\nA =\n', A2)
print('\ntres piezas  c =', c3, '  b =', b3, '\nA =\n', A3)

# Con tres piezas y tres recursos A es CUADRADA, y la prueba de tamaños se queda
# callada: callar no es aprobar. Toca revisar contra la bitácora, y lo que se
# revisa es un renglón y una columna.
assert A3[1, 0] == 2, A3[1, 0]                    # A_21: polímero por filtro, 2 kg
assert list(A3[:, 2]) == [1, 2, 3], A3[:, 2]      # la columna del sello, su receta
assert np.array_equal(A3[:, :2], A2), (A3, A2)    # las dos primeras columnas son la clase 1
print('\nel renglón del polímero empieza en', A3[1, 0], 'kg por filtro;',
      'la columna del sello es', A3[:, 2],
      '\ny las dos primeras columnas de A son, tal cual, el modelo de la clase 1')

## 2 · El solver contra las dos trazas a mano

La clase caminó dos veces a mano: sobre el polígono, del origen a $(8,2)$, y
sobre el poliedro con sello, del origen a $(5,2,3)$. La pregunta es si el solver
termina donde terminaron ellas.

Ojo con lo que se compara: **el destino, no la ruta**. `linprog` no promete pasar
por los mismos vértices, y *De esquina en esquina* ya avisó que de otra ruta se
promete el valor y no el vértice. Aquí los dos óptimos son únicos, así que el
destino sí tiene que coincidir; si alguno tuviera empate, esta comparación sería
tramposa.

In [ ]:
# === Celda 3 · linprog, y el signo de vuelta ===================

def resolver(c, A, b):
    """Maximiza c·x sujeto a A x <= b, x >= 0. Devuelve (x, valor, estado)."""
    r = linprog(c=-np.asarray(c, float), A_ub=np.asarray(A, float),
                b_ub=np.asarray(b, float), method='highs')
    if r.status != 0:                       # 2 = infactible, 3 = no acotado
        return None, None, r.message.split('.')[0]
    return r.x, -r.fun, 'óptimo'            # ojo con el signo de vuelta

a_mano = [('dos piezas', c2, A2, b2, [8, 2],    38),
          ('con sello',  c3, A3, b3, [5, 2, 3], 41)]

for nombre, c, A, b, fin, vale in a_mano:
    x, z, estado = resolver(c, A, b)
    coincide = np.allclose(x, fin, atol=TOL) and np.isclose(z, vale)
    print(f'{nombre:11} {estado}: {fmt(x)} con {z:g} créditos   '
          f'la traza a mano terminó en {fmt(fin)} con {vale}   '
          f'-> {"coincide" if coincide else "NO COINCIDE"}')

for nombre, c, A, b, fin, vale in a_mano:
    x, z, _ = resolver(c, A, b)
    assert np.allclose(x, fin, atol=TOL), (nombre, x)
    assert np.isclose(z, vale), (nombre, z)
print('\nlos dos destinos coinciden: (8, 2) con 38, y (5, 2, 3) con 41')

## 3 · Los ocho vértices, contados por la máquina

*Cuando se acaba el dibujo* tabula ocho vértices con lo que se acaba en cada uno,
y *De esquina en esquina* dice quién es vecino de quién. Las dos cosas son
cuentas, y aquí se hacen.

El método es el de la clase 1 subido una dimensión: cruzar restricciones de tres
en tres, quedarse con los cruces que cumplen todo lo demás, y tirar los
repetidos. Con seis restricciones —tres recursos y tres $x_j\ge0$— son
$\binom{6}{3}=20$ tríos, y de los veinte sobreviven ocho.

Sobre la vecindad hay que declarar una licencia. *De esquina en esquina* decide
vecinos **contando** activas compartidas, y contar solo alcanza porque ninguno de
estos dos poliedros es degenerado: en cada vértice se cumplen con igualdad
exactamente $n$ restricciones, ni una más. La celda comprueba primero eso, y después compara
el conteo contra el criterio bueno —que las compartidas sean independientes—,
que es una cuenta de rango. Donde los dos coinciden, contar está permitido.

In [ ]:
# === Celda 4 · Los vértices, por enumeración ===================
# El sistema completo son las filas de recurso más una fila por cada x_j >= 0,
# escrita como -x_j <= 0 para que TODO quede con <=.

def sistema(A, b):
    n = A.shape[1]
    return np.vstack([A, -np.eye(n)]), np.concatenate([b, np.zeros(n)])

def enumerar(A, b):
    """Cruza n restricciones, quédate con los cruces que cumplen el resto.
    Devuelve los vértices y cuántos tríos no se cortaron en un punto."""
    F, L = sistema(A, b)
    n = A.shape[1]
    V, singulares = [], 0
    for idx in combinations(range(len(F)), n):
        M = F[list(idx)]
        if abs(np.linalg.det(M)) < TOL:          # fronteras que no se cortan en un punto
            singulares += 1
            continue
        x = np.linalg.solve(M, L[list(idx)])
        if np.all(F @ x <= L + TOL) and not any(np.allclose(x, w, atol=TOL) for w in V):
            V.append(x)
    return V, singulares

def activas(A, b, x, nombres):
    F, L = sistema(A, b)
    return [nombres[i] for i in range(len(L)) if abs(F[i] @ x - L[i]) < TOL]

nombres2 = recursos + ['x₁=0', 'x₂=0']
nombres3 = recursos + ['x₁=0', 'x₂=0', 'x₃=0']

V3, singulares3 = enumerar(A3, b3)
V3.sort(key=lambda v: c3 @ v)

print(f'{len(list(combinations(range(6), 3)))} tríos que revisar, '
      f'{singulares3} sin punto de corte, y {len(V3)} vértices que sobreviven\n')
print(f'{"plan":16}{"vale":>7}   qué se acaba ahí')
for v in V3:
    print(f'{fmt(v):16}{c3 @ v:7g}   ' + ', '.join(activas(A3, b3, v, nombres3)))

In [ ]:
# === Celda 5 · Contra la tabla de la unidad ====================
# La tabla de *Cuando se acaba el dibujo*, copiada a mano de esa página, y
# comparada renglón por renglón con lo que acaba de salir. El (9/2, 0, 9/2) de la
# página se escribe aquí 4.5, y su 81/2 es 40.5.

tabla_de_la_pagina = [
    ([0,   0, 0  ],  0,   ['x₁=0', 'x₂=0', 'x₃=0']),
    ([0,   9, 0  ], 27,   ['energía', 'x₁=0', 'x₃=0']),
    ([0,   0, 6  ], 30,   ['energía', 'x₁=0', 'x₂=0']),
    ([2,   8, 0  ], 32,   ['horas', 'energía', 'x₃=0']),
    ([9,   0, 0  ], 36,   ['polímero', 'x₂=0', 'x₃=0']),
    ([8,   2, 0  ], 38,   ['horas', 'polímero', 'x₃=0']),
    ([4.5, 0, 4.5], 40.5, ['polímero', 'energía', 'x₂=0']),
    ([5,   2, 3  ], 41,   ['horas', 'polímero', 'energía']),
]

assert len(V3) == 8, len(V3)
for v, (plan, vale, act) in zip(V3, tabla_de_la_pagina):
    assert np.allclose(v, plan, atol=TOL), (fmt(v), plan)
    assert np.isclose(c3 @ v, vale), (fmt(v), c3 @ v, vale)
    assert activas(A3, b3, v, nombres3) == act, (fmt(v), activas(A3, b3, v, nombres3), act)
print('los ocho vértices, sus valores y sus activas coinciden uno por uno con la '
      'tabla de Cuando se acaba el dibujo')

# Y el plan de la clase 1 sigue ahí, con un sello de más y un renglón menos de
# valor: es vértice, vale 38, y ya no gana.
assert any(np.allclose(v, [8, 2, 0], atol=TOL) for v in V3), V3
assert np.isclose(c3 @ np.array([8., 2, 0]), 38), c3 @ np.array([8., 2, 0])
assert np.isclose(c3 @ np.array([5., 2, 3]), 41), c3 @ np.array([5., 2, 3])
print('el plan de la clase 1, (8, 2, 0), sigue siendo vértice: vale 38 contra 41')

In [ ]:
# === Celda 6 · Quién es vecino de quién ========================
# Dos criterios. El barato es el de *De esquina en esquina*: compartir n-1
# activas. El bueno
# pide además que esas n-1 sean independientes, que es una cuenta de rango.
# Contar alcanza SOLO si ningún vértice es degenerado, así que eso se comprueba
# antes: donde sobran activas, el conteo empieza a mentir.

def cuales_activas(A, b, x):
    F, L = sistema(A, b)
    return {i for i in range(len(L)) if abs(F[i] @ x - L[i]) < TOL}

def vecinos_contando(A, b, v, w):
    return len(cuales_activas(A, b, v) & cuales_activas(A, b, w)) == A.shape[1] - 1

def vecinos_por_rango(A, b, v, w):
    F, _ = sistema(A, b)
    compartidas = sorted(cuales_activas(A, b, v) & cuales_activas(A, b, w))
    if not compartidas:
        return False
    return np.linalg.matrix_rank(F[compartidas], tol=TOL) == A.shape[1] - 1

V2, singulares2 = enumerar(A2, b2)
V2.sort(key=lambda v: c2 @ v)

for etiqueta, A, b, V in [('polígono', A2, b2, V2), ('poliedro', A3, b3, V3)]:
    n = A.shape[1]
    degenerados = [fmt(v) for v in V if len(cuales_activas(A, b, v)) != n]
    assert not degenerados, degenerados
    pares = list(combinations(range(len(V)), 2))
    discrepan = [(fmt(V[i]), fmt(V[j])) for i, j in pares
                 if vecinos_contando(A, b, V[i], V[j]) != vecinos_por_rango(A, b, V[i], V[j])]
    assert not discrepan, discrepan
    grados = [sum(1 for w in V if vecinos_por_rango(A, b, v, w)) for v in V]
    assert grados == [n] * len(V), grados
    print(f'{etiqueta}: {len(V)} vértices, ninguno con más de {n} activas; '
          f'contar y calcular el rango dan lo mismo en los {len(pares)} pares; '
          f'cada vértice tiene {n} vecinos')

print('\nvecinos de cada vértice del poliedro, y lo que valen:')
for v in V3:
    vec = sorted([w for w in V3 if vecinos_por_rango(A3, b3, v, w)], key=lambda w: c3 @ w)
    print(f'  {fmt(v):16} -> ' + ' · '.join(f'{fmt(w)}={c3 @ w:g}' for w in vec))

In [ ]:
# === Celda 7 · Simplex, caminando por vecinos ==================
# Las nueve líneas de *De esquina en esquina*, con su regla de desempate: entre
# vecinos que empatan, el de menor x₁, y si también empatan ahí, el de menor x₂.
# La rama del problema no acotado no está: aquí nunca se ejecuta.

def simplex(c, A, b, V, inicio):
    val = lambda x: float(c @ x)
    actual = next(v for v in V if np.allclose(v, inicio, atol=TOL))
    camino = [actual]
    while True:
        mejoran = [w for w in V
                   if vecinos_por_rango(A, b, actual, w) and val(w) > val(actual) + TOL]
        if not mejoran:
            return camino
        tope = max(val(w) for w in mejoran)
        empatan = sorted([w for w in mejoran if val(w) > tope - TOL], key=tuple)
        actual = empatan[0]
        camino.append(actual)

def traza(c, A, b, V, inicio, titulo):
    camino = simplex(c, A, b, V, inicio)
    print(titulo)
    for k, v in enumerate(camino):
        vec = sorted([w for w in V if vecinos_por_rango(A, b, v, w)], key=lambda w: c @ w)
        sigo = (f'me voy a {fmt(camino[k + 1])}' if k + 1 < len(camino)
                else 'ninguno mejora: paro')
        print(f'  {fmt(v):14}={c @ v:5g}   vecinos: '
              + ' · '.join(f'{fmt(w)}={c @ w:g}' for w in vec) + f'   {sigo}')
    print(f'  -> {len(camino) - 1} pivotes, {len(camino)} vértices de {len(V)}\n')
    return camino

t_origen  = traza(c2, A2, b2, V2, [0, 0],    'polígono de dos piezas, desde el origen')
t_esquina = traza(c2, A2, b2, V2, [0, 9],    'el mismo polígono, desde (0, 9): el ejercicio de esa misma página')
t_sello   = traza(c3, A3, b3, V3, [0, 0, 0], 'poliedro con sello, desde el origen')

assert len(t_origen) == 3 and np.allclose(t_origen, [[0, 0], [9, 0], [8, 2]], atol=TOL)
assert len(t_esquina) == 3 and np.allclose(t_esquina, [[0, 9], [2, 8], [8, 2]], atol=TOL)
assert len(t_sello) == 4 and np.allclose(
    t_sello, [[0, 0, 0], [9, 0, 0], [4.5, 0, 4.5], [5, 2, 3]], atol=TOL)
print('las tres trazas son las de las páginas, con sus dos, dos y tres pivotes')

## 4 · El rango de las horas

*Cuánto vale una hora más* tabula el valor óptimo hora por hora, de 8 a 14, y
dice que la hora vale 2 mientras haya **entre 9 y 12** horas. Esa frase tiene dos
extremos, y los dos son codos: en 9 la pendiente pasa de 4 a 2, y en 12 pasa de 2
a 0.

El notebook 1 movió la energía y dibujó un codo. Aquí se mueven las horas, y hay
dos. En un codo no hay derivada: por la izquierda de 12 el valor sube 2 por hora
y por la derecha no sube nada, así que ahí no hay un número, hay dos. Por eso un
precio sombra se dice siempre con su rango pegado, y nunca a secas.

In [ ]:
# === Celda 8 · El valor óptimo, hora por hora ==================
# La columna «ganó esa hora» es una diferencia HACIA ATRÁS: lo que vale tener h
# horas menos lo que valía tener h-1. Por eso la tabla empieza en 8 pero la
# cuenta necesita el valor en 7. Hacia adelante saldría otra fila, y una más
# corta.

horas  = np.arange(7, 15)
valor  = np.array([resolver(c2, A2, [h, b2[1], b2[2]])[1] for h in horas])
planes = [resolver(c2, A2, [h, b2[1], b2[2]])[0] for h in horas]
gano   = np.diff(valor)                      # siete números, de 8 a 14

print(f'{"horas":>6}{"óptimo":>9}{"mejor plan":>14}{"ganó esa hora":>16}')
for k, h in enumerate(horas):
    if h < 8:
        continue
    print(f'{h:6d}{valor[k]:9g}{fmt(planes[k]):>14}{gano[k - 1]:16g}')

assert np.allclose(valor[1:], [32, 36, 38, 40, 42, 42, 42]), valor
assert np.allclose(gano, [4, 4, 2, 2, 2, 0, 0]), gano
print('\nla fila completa es 4 4 2 2 2 0 0, y por eso la hora vale 2 entre las 9 y las 12')

In [ ]:
# === Celda 9 · Los dos codos, dibujados ========================
# La malla incluye 9 y 12 exactos, que es donde están los codos, y un codo se
# reconoce comparando la pendiente de su izquierda con la de su derecha.

malla = np.linspace(8, 14, 121)              # paso de 0.05, con 9.0 y 12.0 dentro
z     = np.array([resolver(c2, A2, [h, b2[1], b2[2]])[1] for h in malla])
pend  = np.round(np.diff(z) / np.diff(malla), 6)
codos = [malla[i] for i in range(1, len(malla) - 1) if pend[i - 1] != pend[i]]

# Los tramos entre codos, con la pendiente medida a la mitad de cada uno.
centros = (malla[:-1] + malla[1:]) / 2
bordes  = [malla[0]] + codos + [malla[-1]]
tramos  = [(h0, h1, pend[np.argmin(np.abs(centros - (h0 + h1) / 2))])
           for h0, h1 in zip(bordes, bordes[1:])]

fig, (arriba, abajo) = plt.subplots(2, 1, figsize=(7.6, 6.6), sharex=True,
                                    gridspec_kw={'height_ratios': [2, 1]})
arriba.plot(malla, z, lw=2.6, color='tab:purple')
arriba.plot(10, 38, 'o', ms=9, color='crimson')
arriba.annotate('el modelo tal cual:\n10 horas, 38 créditos', (10, 38),
                textcoords='offset points', xytext=(10, -34), fontsize=10, color='crimson')
altura_rotulo = z.min() + 0.12 * (z.max() - z.min())
for h in codos:
    for ax in (arriba, abajo):
        ax.axvline(h, ls='--', lw=1, color='crimson')
    arriba.text(h + 0.12, altura_rotulo, f'codo en {h:g}', fontsize=11, color='crimson')

for h0, h1, p in tramos:                      # el precio sombra, tramo por tramo
    abajo.plot([h0, h1], [p, p], lw=2.6, color='tab:purple')
    abajo.annotate(f'{p:g} por hora', ((h0 + h1) / 2, p), textcoords='offset points',
                   xytext=(0, 8), ha='center', fontsize=10, color='tab:purple')
for h in codos:                               # en el codo no hay un número: hay dos
    for p in {p for h0, h1, p in tramos if h in (h0, h1)}:
        abajo.plot(h, p, 'o', ms=7, mfc='white', mec='crimson', zorder=5)

arriba.set(ylabel='créditos del mejor plan',
           title='dos codos: la hora vale 4, luego 2, y al final nada')
abajo.set(xlabel='horas de impresora disponibles', ylabel='lo que vale una hora más',
          ylim=(-1, 5.6))
for ax in (arriba, abajo):
    ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

assert len(codos) == 2 and np.allclose(codos, [9, 12]), codos
assert np.allclose([p for _, _, p in tramos], [4, 2, 0]), tramos
print('dos codos, en 9 y en 12, y tres pendientes: 4, luego 2, luego 0')

## 5 · Ahora tú: el taller

Otro rincón de la nave, y esta vez lo único que se pide es **escribir los tres
datos**. La historia no está resuelta en ninguna página.

> **Bitácora del taller.** En esta parada el depósito nos compra tres piezas:
> juntas a 7 créditos, bridas a 4 y pernos a 6.
>
> Del torno quedan 10 horas, y de acero quedan 15 kilos.
>
> La junta se lleva 1 hora y 3 kilos, la brida 2 horas y 1 kilo, y el perno 3
> horas y 2 kilos.

Escribe $c$, $A$ y $b$ en la celda de abajo, **en el orden en que la bitácora
nombra las cosas**: junta, brida y perno para las columnas; torno y acero para
los renglones. La celda no te enseña la respuesta: revisa dato por dato y te dice
cuál no cuadra todavía.

Antes de teclear, mira la forma que va a tener $A$. Aquí hay **más piezas que
recursos**, así que no es cuadrada y la prueba de tamaños de *El modelo como
matriz* vuelve a servir para algo.

In [ ]:
# === Celda 10 · Tus tres datos, revisados ======================
# Está sembrada con los números DEL SELLO, que son de otro problema: corre la
# celda tal cual y lo primero que falla es el tamaño. Cámbialos por los del
# taller y vuelve a correr.

c_taller = np.array([4, 3, 5])                            # <- créditos por pieza
A_taller = np.array([[1, 1, 1], [2, 1, 2], [1, 2, 3]])    # <- un renglón por recurso
b_taller = np.array([10, 18, 18])                         # <- de cuánto dispones

# La respuesta no está escrita aquí: está resumida en huellas, para que revisar no
# sea leer. Cada huella resume un dato, y dos datos distintos no dan la misma.
import hashlib
huella = lambda v: hashlib.sha256(
    ','.join(f'{x:g}' for x in np.asarray(v, float).ravel()).encode()).hexdigest()[:8]

ESPERADO = {'c':                    '85f9cf03',
            'A, renglón del torno': '8a6ae151',
            'A, renglón del acero': 'edb794d6',
            'b':                    '1638632c'}

def revisar(c, A, b):
    c, b, A = np.ravel(c), np.ravel(b), np.atleast_2d(A)
    print(f'A tiene {A.shape[0]} renglones y {A.shape[1]} columnas: '
          f'{A.shape[0]} recursos y {A.shape[1]} piezas')
    if A.shape != (2, 3) or len(b) != A.shape[0] or len(c) != A.shape[1]:
        print('los tamaños todavía no cuadran: la bitácora trae dos recursos y tres piezas')
        return
    mios = {'c': c, 'A, renglón del torno': A[0], 'A, renglón del acero': A[1], 'b': b}
    mal = [k for k, v in mios.items() if huella(v) != ESPERADO[k]]
    for k in mios:
        print(f'  {k:22} {"cuadra" if k not in mal else "todavía no"}')
    if mal:
        print('\nvuelve a la bitácora por: ' + ', '.join(mal))
        return
    x, valor, estado = resolver(c, A, b)
    print(f'\nlos tres datos cuadran. {estado}: {fmt(x)} y {valor:g} créditos')
    for k, nombre in enumerate(['torno', 'acero']):
        gasta = A[k] @ x
        print(f'  {nombre:6} gastas {gasta:g} de {b[k]:g}   '
              + ('se acaba' if abs(b[k] - gasta) < TOL else f'sobran {b[k] - gasta:g}'))
    print('\nY algo que se lee en el plan: una de las tres piezas se queda en cero.\n'
          'Con dos recursos no puede pasar otra cosa. Un vértice necesita tres\n'
          'restricciones activas y solo hay dos renglones de recurso, así que al\n'
          'menos una de las tres tiene que ser una x_j = 0.')

revisar(c_taller, A_taller, b_taller)